# Transformer Encoder Classifier -- Majority-Token Task (Training)

ex.13: a full Transformer, trained on a dataset, not just a mechanics
demo. Builds on `code/transformers/transformer_block.py`
(`TransformerEncoderBlock`: multi-head self-attention, residual
connections, LayerNorm, position-wise MLP -- Vaswani et al., 2017,
Sec. 3.1/3.2.2/3.3) by stacking several of these blocks into an
encoder-only classifier and training it end to end.

**Task -- "majority token"** (`code/transformers/majority_token_dataset.py`):
given a sequence of $L$ integer tokens from a vocabulary of size $V$,
predict the value that is the (constructed-to-be-unique) majority
element of the sequence. This is deliberately *not* yet an image-patch
task (ViT is a later, separate exercise) -- it is chosen because
solving it genuinely requires attending to every position and
aggregating, which is exactly what self-attention is for, and it is
easy to visualize: a trained model's `[CLS]` token should attend most
to the positions holding the majority value.

**Model** (`code/transformers/majority_token_classifier.py`,
`MajorityTokenTransformerClassifier`):
```
tokens [B,L] int64
  -> token embedding                  [B,L,D]
  -> prepend learned CLS token        [B,L+1,D]
  -> + learned positional embedding   [B,L+1,D]
  -> N x TransformerEncoderBlock      [B,L+1,D]  (shape-preserving)
  -> take CLS position (index 0)      [B,D]
  -> linear classifier head           [B,V]  = logits
```

Per this project's train/evaluate notebook split: **this notebook only
trains and checkpoints** (model + optimizer + epoch + config, saved to
Google Drive, plus a JSON-lines metrics log). Qualitative results
(attention maps, sample predictions, confusion matrix) live in the
companion `majority_token_evaluate.ipynb`, which loads this notebook's
checkpoint -- so new diagnostics can be added there and rerun in
seconds, no retraining required.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

All model/dataset/training code lives in `code/transformers/*.py`, not
duplicated here. Replace `REPO_URL` with this repository's URL if
different.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code", "transformers"))


## 3. Build the dataset and inspect shapes

`make_majority_token_datasets` generates the train and test splits
independently (different seeds), each sample's label guaranteed to be
the sequence's unique majority value by construction (see the
docstring in `majority_token_dataset.py` for the exact argument).

In [ ]:
from majority_token_dataset import make_majority_token_datasets

VOCAB_SIZE = 12
SEQ_LEN = 16
NUM_TRAIN = 20000
NUM_TEST = 4000
SEED = 0

train_dataset, test_dataset = make_majority_token_datasets(VOCAB_SIZE, SEQ_LEN, NUM_TRAIN, NUM_TEST, seed=SEED)
print(f"train_dataset: {len(train_dataset)} samples, sequences {tuple(train_dataset.sequences.shape)}")
print(f"test_dataset:  {len(test_dataset)} samples, sequences {tuple(test_dataset.sequences.shape)}")

sequence, label = train_dataset[0]
print(f"example sequence = {sequence.tolist()}")
print(f"example majority label = {label.item()}")


## 4. Build the model and print shapes end to end

One forward pass on a small synthetic batch, purely to print the shape
at every stage (`[B,L]` -> `[B,L+1,D]` -> ... -> `[B,V]`) before any
real training happens.

In [ ]:
from majority_token_classifier import MajorityTokenTransformerClassifier, print_classifier_shapes

EMBED_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 2
FFN_HIDDEN_DIM = 128

torch.manual_seed(SEED)
model = MajorityTokenTransformerClassifier(VOCAB_SIZE, SEQ_LEN, EMBED_DIM, NUM_HEADS, NUM_LAYERS, FFN_HIDDEN_DIM).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {num_params:,}")

sample_tokens = torch.randint(0, VOCAB_SIZE, (4, SEQ_LEN), device=device)
sample_outputs = model(sample_tokens)
print_classifier_shapes(sample_tokens, sample_outputs)


## 5. Mount Google Drive, persistent checkpoint + log paths

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so the checkpoint and metrics log are written to a
persistent Drive folder instead of `/content`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

CHECKPOINT_PATH = "/content/drive/MyDrive/thesis_data/majority_token/checkpoint.pt"
LOG_PATH = "/content/drive/MyDrive/thesis_data/majority_token/train_log.jsonl"


## 6. Train

`train()` (imported from `majority_token_train.py`, not duplicated
here) runs the full loop: per-epoch train/test cross-entropy loss and
accuracy (test split evaluated with no gradient step, never used for
updates), a JSON-lines metrics log, and a checkpoint (model + optimizer
+ epoch + config) saved every epoch. Set `RESUME = True` to continue
from `CHECKPOINT_PATH` if it already exists (e.g. after a Colab
disconnect) -- the config embedded in the checkpoint is reused, so the
architecture hyperparameters above don't need to be retyped.

In [ ]:
import argparse
from majority_token_train import train

RESUME = False
EPOCHS = 15

train_args = argparse.Namespace(
    vocab_size=VOCAB_SIZE,
    seq_len=SEQ_LEN,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ffn_hidden_dim=FFN_HIDDEN_DIM,
    num_train=NUM_TRAIN,
    num_test=NUM_TEST,
    batch_size=128,
    lr=3e-4,
    epochs=EPOCHS,
    seed=SEED,
    checkpoint_path=CHECKPOINT_PATH,
    log_path=LOG_PATH,
    resume=RESUME,
)
train(train_args)


## 7. Loss / accuracy curves (quantitative result over training)

In [ ]:
from majority_token_evaluate import plot_training_curves

fig = plot_training_curves(LOG_PATH)
import matplotlib.pyplot as plt
plt.show()


## 8. Next step

Open `notebooks/colab/transformers/majority_token_evaluate.ipynb`,
point it at the same `CHECKPOINT_PATH` / `LOG_PATH`, and run the
confusion matrix, sample predictions, and CLS attention-map
visualizations -- no retraining required.